# NaijaDev Studio — N-ATLAS Inference (Kaggle)

A reproducible, **secret-free** demonstration of the inference service used by NaijaDev Studio. The model is `NCAIR1/N-ATLaS`, subject to Hugging Face access approval. This notebook is designed for a Kaggle session with **two T4 GPUs** and internet access.

**Before running:** In Kaggle **Add-ons → Secrets**, configure `HF_TOKEN` (authorized Hugging Face access token) and `NATLAS_API_KEY` (your own strong private API key). Enable access to these secrets for the notebook. Do not print or publish their values. Running this notebook starts a new inference service and a temporary public tunnel; it does not reuse the original session. The tunnel URL is not permanent.

In [ ]:
%pip install -q -U transformers accelerate huggingface_hub safetensors sentencepiece fastapi uvicorn

In [ ]:
import torch
from kaggle_secrets import UserSecretsClient
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.device_count() >= 2, "This configuration requires two GPUs"
secrets_client = UserSecretsClient()
hf_token = secrets_client.get_secret("HF_TOKEN")
api_key = secrets_client.get_secret("NATLAS_API_KEY")
assert hf_token and api_key, "Configure both Kaggle secrets before proceeding"

model_id = "NCAIR1/N-ATLaS"
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id, token=hf_token, dtype=torch.float16,
    device_map="auto", low_cpu_mem_usage=True,
    max_memory={0: "13GiB", 1: "13GiB", "cpu": "8GiB"}
)
model.eval()
print("N-ATLAS loaded; device allocation:", model.hf_device_map)

In [ ]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import secrets as python_secrets

app = FastAPI(title="NaijaDev Studio N-ATLAS Inference")

class PromptRequest(BaseModel):
    language: str
    prompt: str

@app.get("/health")
def health():
    return {"status": "ok", "model": "N-ATLAS"}

@app.post("/generate")
def generate(request: PromptRequest, x_api_key: str = Header(default="")):
    if not python_secrets.compare_digest(x_api_key, api_key):
        raise HTTPException(status_code=401, detail="Unauthorized")
    if request.language not in {"English", "Hausa", "Igbo", "Yoruba"}:
        raise HTTPException(status_code=400, detail="Unsupported language")
    messages = [{"role": "user", "content": f"Answer in {request.language}. {request.prompt}"}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=150, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = output[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(
        generated, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return {"response": response}

In [ ]:
import threading, time, uvicorn

server = uvicorn.Server(uvicorn.Config(
    app, host="127.0.0.1", port=8001, log_level="info"
))
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(3)
print("API started:", server.started)

In [ ]:
import os, subprocess, re, time
cloudflared_path = "/kaggle/working/cloudflared"
if not os.path.exists(cloudflared_path):
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", cloudflared_path
    ], check=True)
    os.chmod(cloudflared_path, 0o755)

log_path = "/kaggle/working/tunnel.log"
with open(log_path, "w") as log:
    tunnel = subprocess.Popen([
        cloudflared_path, "tunnel", "--url",
        "http://127.0.0.1:8001", "--no-autoupdate"
    ], stdout=log, stderr=subprocess.STDOUT)
time.sleep(12)
with open(log_path) as log:
    tunnel_output = log.read()
urls = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", tunnel_output)
print("Tunnel active:", tunnel.poll() is None)
print("Temporary tunnel URL:", urls[0] if urls else "Not found; check tunnel.log")

## Connect your local backend

Set the local `backend/.env` values to the **current** tunnel URL and the same `NATLAS_API_KEY` stored in Kaggle secrets:

```dotenv
NATLAS_API_URL=https://YOUR-CURRENT-TUNNEL.trycloudflare.com
NATLAS_API_KEY=YOUR_PRIVATE_API_KEY
```

The local FastAPI backend sends `x-api-key` in the request header. **Never commit `.env` or paste real tokens into GitHub.** A Cloudflare Quick Tunnel is temporary; Kaggle sessions can stop, and the link can change. This notebook is an example of the working architecture, not a continuously hosted deployment.